# Chapter 10 — Error Handling in Multi-agent Systems · Homework (self-test)

**Goal:** check that you can classify errors, recover locally without amplifying failures, tell an access failure from a valid empty result, design the subagent envelope and the coordinator's policy, isolate subagents, and report coverage honestly.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 75–100 minutes
- Estimated API cost: **$0**. Everything graded runs offline; the one optional live cell sends a request with an unknown model ID, which returns an error and is not billed.
- Prerequisites: the practice notebook, and the venv from [00-prerequisites](../../00-prerequisites/README.md)

**How to use it**
1. Attempt everything **without peeking**. Solutions are at the very bottom, collapsed.
2. Part A: answer 10 exam-style questions in the `answers` cell, then run the grader.
3. Part B: replace each `TODO` stub, then run its checker. Checkers never crash the notebook; they tell you what is wrong.
4. Part C: write a short design answer, then compare it with the rubric.
5. Run the **Scorecard**. Pass mark: **72%** overall (quiz + exercises), chosen to echo the exam's scaled cut score of 720/1000 (the scale is not percent-correct).

In [ ]:
import asyncio
import concurrent.futures
import hashlib
import json
import os
import time
from typing import Awaitable, Callable, Literal

import anthropic
import httpx2
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, Field, model_validator

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
print("anthropic", anthropic.__version__, "| MODEL =", MODEL)

The optional live cell needs a key. Without one it prints "skipped": everything graded is offline.

In [ ]:
HAS_KEY = bool(os.getenv("ANTHROPIC_API_KEY"))
print("live cell enabled" if HAS_KEY else "no ANTHROPIC_API_KEY: the optional live cell will be skipped")

### Helpers (run, don't edit)

The chapter's contract from the README (`Category`, `ToolFailure`, `Finding`, `SubagentResult` with its validator), the `SearchOutcome` model for Exercise 3, the grading bookkeeping, `run_async()` for the async exercise, `make_api_error()` (real SDK exceptions from scripted HTTP responses), and `envelope()` for building test results.

In [ ]:
Category = Literal["transient", "validation", "business", "permission"]


class ToolFailure(Exception):
    """A failure the tool layer has already classified (README 10.5)."""

    def __init__(self, category: Category, retryable: bool, message: str, **extra):
        super().__init__(message)
        self.category, self.retryable, self.extra = category, retryable, extra
        self.attempts = 1


class Finding(BaseModel):
    claim: str
    source_url: str
    relevance: float


class SubagentResult(BaseModel):
    """The structured subagent result envelope (README 10.6)."""
    subagent: str
    topic: str
    status: Literal["success", "empty", "partial", "failed"]
    failure_type: Literal["timeout", "rate_limited", "unavailable", "invalid_input", "permission_denied",
                          "policy_blocked", "parse_error", "budget_exhausted", "internal_error"] | None = None
    error_category: Category | None = None
    retryable: bool | None = None
    attempted: list[str] = Field(default_factory=list)
    attempts: int = 1
    findings: list[Finding] = Field(default_factory=list)
    alternatives: list[str] = Field(default_factory=list)
    coverage_impact: str | None = None
    message: str | None = None

    @model_validator(mode="after")
    def consistent(self):
        failed = self.status in ("partial", "failed")
        if failed and (self.failure_type is None or self.retryable is None):
            raise ValueError("partial/failed results must say what failed and whether a retry can help")
        if not failed and self.failure_type is not None:
            raise ValueError("success/empty results must not carry a failure_type")
        if self.status == "empty" and self.findings:
            raise ValueError("an empty result cannot contain findings")
        if self.status == "partial" and not self.findings:
            raise ValueError("partial means some findings survived; use 'failed' otherwise")
        return self


class SearchOutcome(BaseModel):
    """Exercise 3's return type: one source, one query, one precise outcome."""
    source: str
    query: str
    status: Literal["ok", "empty", "failed"]
    hits: list[dict] = Field(default_factory=list)
    error: str | None = None
    retryable: bool | None = None


Action = Literal["accept", "retry_modified", "accept_partial", "delegate", "escalate", "annotate_gap"]


# ---------- grading bookkeeping ----------
RESULTS: dict[int, bool] = {}
SCORE: dict[str, int] = {}


def _run_check(n: int, check: Callable[[], None]) -> None:
    """Run one checker; never let it crash the notebook."""
    try:
        check()
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")
    except NotImplementedError:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: not implemented yet (the stub still raises NotImplementedError)")
    except AssertionError as exc:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: {exc}")
    except BaseException as exc:  # noqa: BLE001 - report, don't crash (ExceptionGroup, CancelledError, ...)
        RESULTS[n] = False
        print(f"✗ Exercise {n}: your code raised {type(exc).__name__}: {exc}")


def run_async(coro):
    """Run a coroutine from a normal cell, inside Jupyter (already-running loop) or a plain script."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with concurrent.futures.ThreadPoolExecutor(1) as pool:
        return pool.submit(asyncio.run, coro).result()


def make_api_error(status, headers: dict | None = None, error_code: str | None = None) -> anthropic.APIError:
    """A REAL SDK exception, raised by the SDK itself from a scripted HTTP response (no network, no key).
    status: an HTTP status code, or "connect" / "timeout" for transport failures.
    error_code: optional error.details.error_code, e.g. "enforced_spend_limit_reached"."""
    def handler(request):
        if status == "connect":
            raise httpx2.ConnectError("connection reset (scripted)", request=request)
        if status == "timeout":
            raise httpx2.ReadTimeout("read timed out (scripted)", request=request)
        error = {"type": "scripted_error", "message": f"scripted {status}"}
        if error_code:
            error["details"] = {"error_code": error_code}
        return httpx2.Response(status, headers={"request-id": "req_hw", **(headers or {})},
                               json={"type": "error", "request_id": "req_hw", "error": error})

    fake = anthropic.Anthropic(api_key="offline", max_retries=0,
                               http_client=anthropic.DefaultHttpxClient(transport=httpx2.MockTransport(handler)))
    try:
        fake.messages.create(model=MODEL, max_tokens=10, messages=[{"role": "user", "content": "hi"}])
    except anthropic.APIError as err:
        return err
    raise RuntimeError("expected the scripted call to fail")


def envelope(status: str, *, ftype=None, category=None, retryable=None, findings: int = 0,
             alternatives=(), subagent: str = "web_search", topic: str = "music") -> SubagentResult:
    """Build a valid SubagentResult quickly (used by the checkers)."""
    return SubagentResult(
        subagent=subagent, topic=topic, status=status, failure_type=ftype, error_category=category,
        retryable=retryable, alternatives=list(alternatives),
        findings=[Finding(claim=f"claim {i}", source_url=f"https://example.org/{i}", relevance=0.7)
                  for i in range(findings)])


print("helpers ready")

## Part A — Quiz (10 questions)

Pick one letter per question. The questions cover error categories, empty results vs failures, local recovery, envelopes, coverage, limits, and origin isolation.

**Q1. (Customer support agent, MCP tools)** A `search_orders` MCP tool is called with a valid customer ID. The database answers normally: this customer has no orders. What should the tool return?

- A) `isError: true` with `errorCategory: "validation"`, so the agent asks the customer to re-check the ID
- B) A successful result (`isError: false`) with an empty `orders` list, so the agent can say there are no orders on record
- C) `isError: true` with `errorCategory: "transient"` and `isRetryable: true`, since an order might appear later
- D) A JSON-RPC protocol error, because the request produced no data

**Q2. (Multi-agent research system)** Which of these failures should a document-analysis subagent retry **locally** before reporting anything to the coordinator?

- A) A spreadsheet export in a file format the parser does not support
- B) A 403 from the licensed journal archive because the service account's token expired
- C) A search query rejected for a syntax error, re-sent unchanged
- D) A call to the OCR service that was cut off by its 30-second timeout

**Q3. (Tool design)** A `search_case_law` tool rejects queries that use the `NEAR/n` operator, which its backend does not support. Today it returns `isError: true` with the text "Search failed", and the agent resends the identical query four times before giving up. What is the best fix?

- A) Return a structured validation error (`errorCategory: "validation"`, `isRetryable: false`) whose message names the unsupported operator and an accepted form, such as `AND`, so the agent changes the query once instead of repeating it
- B) Mark the error `isRetryable: true` so that the agent's retries at least back off
- C) Return an empty success result so the agent moves on to other sources
- D) Raise `max_retries` on the Claude client so the call eventually goes through

**Q4. (Production reliability)** A coordinator's Messages API calls start failing with `429 rate_limit_error`. The responses carry no `retry-after` header, the SDK's automatic retries also fail, and the errors persist for hours. Your monitoring shows the organization reached its usage tier's monthly spend cap. What should the system do?

- A) Keep retrying every call with exponential backoff capped at 8 seconds until calls succeed
- B) Classify it as a permission error and automatically rotate to another API key
- C) Stop starting new calls, report the affected runs as failed for a non-retryable business reason (spend cap), and alert a human
- D) Switch every subagent to a smaller model, which has its own rate limits

**Q5. (Agent SDK)** A web-search subagent's run ends with `ResultMessage.subtype == "error_max_turns"` after it had gathered 3 of the 8 sources it planned. What should the coordinator do?

- A) Treat the run as completed, because the subagent returned text
- B) Treat it as a failure to report: record a partial or failed envelope with what was completed, then retry with a narrower scope, accept the partial result, or annotate the gap
- C) Remove `max_turns` from all subagents so this cannot happen again
- D) Discard the subagent's output and restart the whole research run from scratch

**Q6. (Agent SDK, multi-agent research system)** Subagents are told to write error details to a log file and to finish with a short prose summary. Final reports often say "no sources found" for topics where the search actually timed out. What is the root cause and fix?

- A) The coordinator's model is too small; switch it to a larger model
- B) Add a `SubagentStop` hook that deletes the log file after each run
- C) Increase every search timeout so timeouts stop happening
- D) Only the subagent's final message reaches the coordinator; put a structured envelope (status, failure type, attempted queries, partial results) in that final message and validate it in the coordinator

**Q7. (Multi-agent research system)** Who should decide the coverage labels (FULL / PARTIAL / NONE) that appear in the final report?

- A) Code in the coordinator, computed from the subagent envelopes and the topic plan, then passed to the synthesis step with instructions to label each section
- B) The synthesis model, based on its overall impression of how strong the evidence looks
- C) Each subagent, which labels its own topic as fully covered once it finishes
- D) Nobody: labels distract readers, so the report should read as complete

**Q8. (Production reliability)** A `create_ticket` tool call times out, the subagent retries it, and the customer ends up with two identical tickets. What is the right fix?

- A) Never retry tool calls that time out
- B) Raise the timeout so the call never times out
- C) Make the write idempotent (an idempotency key, or check whether the ticket exists before retrying) and keep the bounded retry
- D) Have the coordinator delete duplicate tickets in a nightly job

**Q9. (Debugging (integration vs model output))** In a trace, every tool call returned `200` with data, the final response has `stop_reason: "end_turn"`, and the answer quotes a market-size figure that appears in no tool result. Where is the problem, and where does the fix belong?

- A) Integration layer: increase `max_retries` so the missing data is fetched
- B) Error contract: the tool should have returned `isError: true`
- C) Limits: raise `max_tokens` so the model can include more of the tool output
- D) Model output: the claim is unsupported; fix it with prompts that require citing tool results, a validation step that checks claims against sources, or human review

**Q10. (Multi-agent research system)** A news-archive source has failed about 40% of calls for two days. Each research run waits through three 30-second timeouts on it before moving on, which doubles latency. What is the best change?

- A) Raise the news subagent's timeout to 120 seconds
- B) Add a circuit breaker that fails fast with an `unavailable` envelope after repeated failures (with a cool-down and a trial call), and let the coordinator use the other sources and annotate the gap
- C) Silently remove the news source from the plan until someone notices
- D) Retry the news source at both the subagent and the coordinator level so that more calls eventually succeed

In [ ]:
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
    "q9": None,
    "q10": None,
}

In [ ]:
ANSWER_HASHES = {'q1': '96e2e2c94d02b534d4bdc303373df7b704ed4725981fac17549ee985d132c365', 'q2': 'd14b6ae0cbc51f4b715924684653e0d81035a8fe8b79de34ff29eb9db56e2d6b', 'q3': '2501e2fe43d858f1773d414ee14ab0a8cdf2afefbc4ffd20ae5d956e618f3ea3', 'q4': '69a9828043db3a177204403c987facd12c855928421b3aeeb5edce8407e3196c', 'q5': 'fbb43cbbf51767ad8ca178fa6cc4731661f1f1771789b9b7df93fba7ad386e2c', 'q6': '70425e0bfb4a573a030df7909d2bae441e3fdb1e7a565e097f3118231f4f27a9', 'q7': '8722338db4eb12eaae5f6f26c68875af2b115581d770f01f98b9e0e1e6010ac1', 'q8': '89443923f22dc808ba069cfb289117719cbe7d8d36703c74314c6b70fba0371b', 'q9': 'e58517837d895fe6365a5801be9a66d9d1d39f1d863920a84cbed549cfec1954', 'q10': 'c69a9867de9de4cc2e45f4cd6a6128833af714d8993edbeac622795c9c5e20d3'}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_HASHES, key=lambda q: int(q[1:])):
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == ANSWER_HASHES[qid]
        correct += ok
        print(f"{'✓' if ok else '✗'} {qid}: {letter or '(no answer)'}")
    SCORE["quiz"] = correct
    print(f"Quiz: {correct}/{len(ANSWER_HASHES)} = {correct / len(ANSWER_HASHES):.0%}")
    return correct


_ = grade_quiz(answers)

## Part B — Coding exercises (offline)

Each exercise: read the task, replace the stub, run the checker.

### Exercise 1 — Classify Claude API errors

Write `classify_error(err)` returning `(category, retryable)` for an Anthropic SDK exception. The checker passes **real** SDK exceptions, raised by the SDK from scripted HTTP responses (`make_api_error(status, headers)`), so you can inspect them the same way you would in production (`isinstance`, `err.status_code`, `err.response.headers`, `err.body`).

| Error | Category | Retryable |
|---|---|---|
| `APIConnectionError` (includes `APITimeoutError`) | `"transient"` | `True` |
| 429 whose body has `error.details.error_code == "enforced_spend_limit_reached"` (the tier's monthly spend cap: keeps failing until access resumes) | `"business"` | `False` |
| any other 429, with or without a `retry-after` header (a missing header does not prove a spend cap) | `"transient"` | `True` |
| any 5xx (500, 503, 504, 529) | `"transient"` | `True` |
| 401, 403 | `"permission"` | `False` |
| 402 | `"business"` | `False` |
| 409 (conflict: re-read the state, then retry) | `"transient"` | `True` |
| other 4xx (400, 404, 413, 422) | `"validation"` | `False` |
| not an API error at all (e.g. `KeyError`) | re-raise it | |

Tip: 402 has no dedicated exception class in the SDK, and 503 arrives as `InternalServerError`, so branch on `status_code`. Note (not checked): a 400 whose message starts "You have reached your specified ... API usage limits" is a spend limit you set yourself, so in production you would treat it as business too. Theory: [README 10.2](README.md#102-error-categories-and-the-retryable-flag)

In [ ]:
def classify_error(err: Exception) -> tuple[str, bool]:
    """Return (category, retryable) for an Anthropic SDK exception. Re-raise anything that is not an API error."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    def check():
        cases = [
            (400, None, ("validation", False)), (401, None, ("permission", False)),
            (402, None, ("business", False)), (403, None, ("permission", False)),
            (404, None, ("validation", False)), (409, None, ("transient", True)),
            (413, None, ("validation", False)),
            (422, None, ("validation", False)),
            (429, {"retry-after": "3"}, ("transient", True)), (429, None, ("transient", True)),
            (500, None, ("transient", True)), (503, None, ("transient", True)),
            (504, None, ("transient", True)), (529, None, ("transient", True)),
            ("connect", None, ("transient", True)), ("timeout", None, ("transient", True)),
        ]
        for status, headers, want in cases:
            err = make_api_error(status, headers)
            got = classify_error(err)
            label = f"{type(err).__name__} (status {status}{', with retry-after' if headers else ''})"
            assert got == want, f"{label}: expected {want}, got {got}"
        spend_cap = make_api_error(429, None, error_code="enforced_spend_limit_reached")
        got = classify_error(spend_cap)
        assert got == ("business", False), f"429 with error_code enforced_spend_limit_reached: expected ('business', False), got {got}"
        try:
            classify_error(KeyError("source_url"))
        except KeyError:
            pass
        else:
            raise AssertionError("a non-API exception (a bug in our own code) must be re-raised, not classified")
    _run_check(1, check)


check_ex1()

### Exercise 2 — A retry helper that respects the category

Write `retry_call(op, *, max_attempts=3, base_delay=0.5, max_delay=8.0, sleep=time.sleep)` (synchronous, so the checker can pass a fake `sleep`):

- Call `op()`. On success return `(result, attempts)`.
- Retry **only** a `ToolFailure` whose `retryable` is `True`. A non-retryable `ToolFailure` is re-raised immediately (1 call, no sleep). Any other exception (a bug) propagates immediately and is never retried.
- At most `max_attempts` calls in total. When giving up, set `err.attempts` to the number of attempts made and re-raise.
- Before retry *k* (k = 1, 2, ...) call `sleep(min(base_delay * 2 ** (k - 1), max_delay) * jitter)` with `jitter` drawn uniformly from `[0.5, 1.0]`. Never sleep after the last attempt.

`ToolFailure(category, retryable, message)` is defined in the helpers. Theory: [README 10.5](README.md#105-local-recovery-before-propagation)

In [ ]:
def retry_call(op: Callable[[], object], *, max_attempts: int = 3, base_delay: float = 0.5,
               max_delay: float = 8.0, sleep: Callable[[float], None] = time.sleep) -> tuple[object, int]:
    """Call op(); retry ONLY retryable ToolFailures with capped exponential backoff + jitter. Return (result, attempts)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex2() -> None:
    class FakeSleep:
        def __init__(self):
            self.delays: list[float] = []

        def __call__(self, seconds: float) -> None:
            self.delays.append(seconds)

    def scripted(*outcomes):
        calls = []

        def op():
            outcome = outcomes[min(len(calls), len(outcomes) - 1)]
            calls.append(outcome)
            if outcome == "timeout":
                raise ToolFailure("transient", True, "timed out")
            if outcome == "invalid":
                raise ToolFailure("validation", False, "bad date format")
            if outcome == "denied":
                raise ToolFailure("permission", False, "403")
            if outcome == "bug":
                raise KeyError("oops")
            return ["hit"]
        op.calls = calls
        return op

    def check():
        s, op = FakeSleep(), scripted("timeout", "timeout", "ok")
        got = retry_call(op, sleep=s)
        assert got == (["hit"], 3), f"timeout, timeout, ok: expected (['hit'], 3), got {got}"
        assert len(s.delays) == 2, f"expected 2 sleeps before the 2 retries, got {s.delays}"
        assert 0.25 <= s.delays[0] <= 0.5 and 0.5 <= s.delays[1] <= 1.0, \
            f"delays should be base*2**(k-1) times a jitter factor in [0.5, 1.0]: got {s.delays}"

        for outcome in ("invalid", "denied"):
            s, op = FakeSleep(), scripted(outcome)
            try:
                retry_call(op, sleep=s)
                raise AssertionError(f"{outcome}: the ToolFailure must be re-raised")
            except ToolFailure as err:
                assert len(op.calls) == 1 and not s.delays, \
                    f"{outcome} is not retryable: expected 1 call and no sleep, got {len(op.calls)} calls, sleeps {s.delays}"
                assert err.attempts == 1, f"set err.attempts on the re-raised error (expected 1, got {err.attempts})"

        s, op = FakeSleep(), scripted("timeout")
        try:
            retry_call(op, max_attempts=4, sleep=s)
            raise AssertionError("a persistent timeout must be re-raised after max_attempts")
        except ToolFailure as err:
            assert len(op.calls) == 4, f"max_attempts=4 means 4 calls in total, got {len(op.calls)}"
            assert err.attempts == 4, f"err.attempts should be 4, got {err.attempts}"
            assert len(s.delays) == 3, f"no sleep after the last attempt: expected 3 sleeps, got {len(s.delays)}"

        s, op = FakeSleep(), scripted("timeout", "timeout", "ok")
        retry_call(op, base_delay=4, max_delay=5, sleep=s)
        assert 2 <= s.delays[0] <= 4 and 2.5 <= s.delays[1] <= 5, \
            f"cap each delay at max_delay before jitter (base 4, max 5): got {s.delays}"

        s, op = FakeSleep(), scripted("bug")
        try:
            retry_call(op, sleep=s)
            raise AssertionError("a non-ToolFailure exception must propagate")
        except KeyError:
            assert len(op.calls) == 1, "a bug (non-ToolFailure exception) must not be retried"

        firsts = set()
        for _ in range(20):
            s = FakeSleep()
            retry_call(scripted("timeout", "ok"), sleep=s)
            firsts.add(round(s.delays[0], 6))
        assert len(firsts) > 1, "add random jitter, so that many clients don't retry in lockstep"
    _run_check(2, check)


check_ex2()

### Exercise 3 — Access failure vs valid empty result

Write `search_outcome(source, query, backend, canary_query=None)` returning a `SearchOutcome` (helpers). `backend(source, query)` returns a list of hits or raises.

- `TimeoutError` → `status="failed"`, `retryable=True`, an `error` message.
- `PermissionError` → `status="failed"`, `retryable=False`, an `error` message.
- Hits → `status="ok"` with the hits. Do **not** run the canary in this case.
- No hits and no `canary_query` → `status="empty"`.
- No hits and a `canary_query`: call `backend(source, canary_query)`. If it returns hits, the source is healthy → `status="empty"`. If it returns nothing **or times out**, the emptiness can't be trusted → `status="failed"`, `retryable=True`, and an `error` that mentions the canary.

Theory: [README 10.3](README.md#103-access-failure-vs-valid-empty-result)

In [ ]:
def search_outcome(source: str, query: str, backend: Callable[[str, str], list[dict]],
                   canary_query: str | None = None) -> SearchOutcome:
    """Call backend(source, query) and report ok / empty / failed precisely (see the rules above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> None:
    calls = []

    def backend(source: str, query: str) -> list[dict]:
        calls.append((source, query))
        if source == "down":
            raise TimeoutError("no answer in 10s")
        if source == "locked":
            raise PermissionError("403 from the licensed archive")
        if source == "full":
            return [{"title": "a"}, {"title": "b"}]
        if source == "quiet":                       # healthy, nothing matches
            return [{"title": "canary"}] if query == "CANARY" else []
        if source == "canary_hangs":
            if query == "CANARY":
                raise TimeoutError("canary timed out")
            return []
        return []                                   # "broken": empty for everything

    def check():
        r = search_outcome("full", "q", backend, canary_query="CANARY")
        assert r.status == "ok" and len(r.hits) == 2, f"full: expected ok with 2 hits, got {r.status} {r.hits}"
        assert ("full", "CANARY") not in calls, "don't run the canary when the real query returned hits"

        r = search_outcome("quiet", "q", backend, canary_query="CANARY")
        assert r.status == "empty", f"quiet + canary answering: a valid empty result, got {r.status}"
        r = search_outcome("quiet", "q", backend)
        assert r.status == "empty", f"empty without a canary: report empty, got {r.status}"

        r = search_outcome("broken", "q", backend, canary_query="CANARY")
        assert r.status == "failed" and r.retryable is True, \
            f"broken source (canary also empty): expected failed + retryable, got {r.status}, retryable={r.retryable}"
        assert r.error and "canary" in r.error.lower(), f"say in `error` that the canary also came back empty, got {r.error!r}"

        r = search_outcome("canary_hangs", "q", backend, canary_query="CANARY")
        assert r.status == "failed", f"empty + canary timed out: the emptiness is not verified -> failed, got {r.status}"

        r = search_outcome("down", "q", backend)
        assert r.status == "failed" and r.retryable is True and r.error, \
            f"timeout: expected failed, retryable=True and an error message, got {r.status}, {r.retryable}, {r.error!r}"
        assert not r.hits, "a failed search has no hits"

        r = search_outcome("locked", "q", backend)
        assert r.status == "failed" and r.retryable is False, \
            f"permission error: expected failed and retryable=False, got {r.status}, retryable={r.retryable}"
    _run_check(3, check)


check_ex3()

### Exercise 4 — The coordinator's recovery policy

Write the coordinator's recovery policy from README 10.6 yourself, without looking. The subtle part is rule 3: a **validation** failure (a bad query) has `retryable: false` because it cannot succeed **as sent**, yet it is worth one modified retry when the subagent suggests a concrete reformulation. Write `decide_v2(r, retries_left, has_backup)` that applies these rules **in order**:

1. `status` is `"success"` or `"empty"` → `"accept"`
2. `error_category == "permission"` → `"escalate"`
3. (`retryable` is true **or** `error_category == "validation"`) **and** `retries_left > 0` **and** `alternatives` is not empty → `"retry_modified"`
4. `status == "partial"` → `"accept_partial"`
5. `has_backup` → `"delegate"`
6. otherwise → `"annotate_gap"`

Note what rule 3 does **not** include: a business-rule block (`error_category == "business"`) is never retried, even when alternatives exist. Theory: [README 10.6](README.md#106-the-structured-subagent-result)

In [ ]:
def decide_v2(r: SubagentResult, retries_left: int, has_backup: bool) -> Action:
    """The coordinator's recovery policy, including the validation-retry rule (see the ordered rules above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    alt = ["narrower query: 'AI film scoring tools'"]
    cases = [
        ("valid empty result", envelope("empty"), 2, True, "accept"),
        ("success", envelope("success", findings=3), 0, False, "accept"),
        ("403 with alternatives", envelope("failed", ftype="permission_denied", category="permission",
                                            retryable=False, alternatives=alt), 2, True, "escalate"),
        ("timeout + alternative", envelope("partial", ftype="timeout", category="transient", retryable=True,
                                           findings=1, alternatives=alt), 1, False, "retry_modified"),
        ("bad query + reformulation", envelope("failed", ftype="invalid_input", category="validation",
                                               retryable=False, alternatives=alt), 1, False, "retry_modified"),
        ("bad query, no retries left", envelope("failed", ftype="invalid_input", category="validation",
                                                retryable=False, alternatives=alt), 0, False, "annotate_gap"),
        ("timeout, no alternatives", envelope("failed", ftype="timeout", category="transient", retryable=True),
         2, True, "delegate"),
        ("partial, budget spent", envelope("partial", ftype="timeout", category="transient", retryable=True,
                                           findings=1, alternatives=alt), 0, True, "accept_partial"),
        ("policy block, no backup", envelope("failed", ftype="policy_blocked", category="business",
                                             retryable=False, alternatives=alt), 2, False, "annotate_gap"),
    ]

    def check():
        for label, r, retries_left, backup, want in cases:
            got = decide_v2(r, retries_left, backup)
            hint = ""
            if label == "bad query + reformulation" and got != want:
                hint = " (a validation failure is not retryable AS SENT, but a modified request can succeed)"
            if label == "policy block, no backup" and got != want:
                hint = " (a business-rule block must never be retried, even with alternatives)"
            assert got == want, f"{label} (retries_left={retries_left}, backup={backup}): expected {want!r}, got {got!r}{hint}"
    _run_check(4, check)


check_ex4()

### Exercise 5 — Coverage annotations, computed in code

Write `coverage_report(required_topics, plan, results)` returning one dict per **required** topic, in the order of `required_topics`: `{"topic", "label", "gaps", "heading"}`.

- A topic missing from `plan` → `label="NONE"`, `gaps=["never planned"]`.
- Otherwise, `gaps` lists `"<subagent>: <failure_type>"` for every `partial` or `failed` envelope of the topic (in `results` order), then `"<subagent>: never ran"` for each planned subagent with no envelope (in plan order). An `empty` envelope is **not** a gap.
- `label`: `"NONE"` if the topic has no envelopes; `"FULL"` if there are no gaps; `"PARTIAL"` if at least one envelope is `success`, `empty` or `partial`; otherwise `"NONE"`.
- `heading`: `"<topic> (FULL COVERAGE)"`, `"<topic> (PARTIAL COVERAGE: <gaps joined with '; '>)"` or `"<topic> (NO COVERAGE: <gaps joined with '; '>)"`.

Theory: [README 10.9](README.md#109-coverage-annotations-in-the-final-synthesis)

In [ ]:
def coverage_report(required_topics: list[str], plan: dict[str, list[str]],
                    results: list[SubagentResult]) -> list[dict]:
    """One dict per required topic: {"topic", "label", "gaps", "heading"} (see the rules above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    plan = {"visual art": ["web_search", "documents"], "music": ["web_search", "documents"],
            "literature": ["web_search", "documents"], "film": ["web_search"],
            "sculpture": ["web_search", "documents"], "dance": ["web_search", "documents"]}
    required = ["visual art", "music", "literature", "film", "games", "sculpture", "dance"]
    E = envelope
    results = [
        E("success", findings=3, subagent="web_search", topic="visual art"),
        E("success", findings=2, subagent="documents", topic="visual art"),
        E("failed", ftype="timeout", category="transient", retryable=True, subagent="web_search", topic="music"),
        E("success", findings=2, subagent="documents", topic="music"),
        E("empty", subagent="web_search", topic="literature"),
        E("success", findings=4, subagent="documents", topic="literature"),
        E("partial", ftype="timeout", category="transient", retryable=True, findings=1,
          subagent="web_search", topic="sculpture"),
        E("failed", ftype="permission_denied", category="permission", retryable=False,
          subagent="documents", topic="sculpture"),
        E("failed", ftype="unavailable", category="transient", retryable=True, subagent="web_search", topic="dance"),
        E("failed", ftype="timeout", category="transient", retryable=True, subagent="documents", topic="dance"),
    ]
    want = {
        "visual art": ("FULL", [], "visual art (FULL COVERAGE)"),
        "music": ("PARTIAL", ["web_search: timeout"], "music (PARTIAL COVERAGE: web_search: timeout)"),
        "literature": ("FULL", [], "literature (FULL COVERAGE)"),
        "film": ("NONE", ["web_search: never ran"], "film (NO COVERAGE: web_search: never ran)"),
        "games": ("NONE", ["never planned"], "games (NO COVERAGE: never planned)"),
        "sculpture": ("PARTIAL", ["web_search: timeout", "documents: permission_denied"],
                      "sculpture (PARTIAL COVERAGE: web_search: timeout; documents: permission_denied)"),
        "dance": ("NONE", ["web_search: unavailable", "documents: timeout"],
                  "dance (NO COVERAGE: web_search: unavailable; documents: timeout)"),
    }

    def check():
        rows = coverage_report(required, plan, results)
        assert isinstance(rows, list) and [r.get("topic") for r in rows] == required, \
            f"return one dict per REQUIRED topic, in order {required}; got topics {[r.get('topic') for r in rows] if isinstance(rows, list) else rows}"
        for row in rows:
            label, gaps, heading = want[row["topic"]]
            hint = ""
            if row["topic"] == "literature" and row["label"] != label:
                hint = " ('0 results' is an answer, not a gap)"
            if row["topic"] == "games" and row["label"] != label:
                hint = " (a required topic that was never planned has NO coverage)"
            assert row["label"] == label, f"{row['topic']}: expected label {label}, got {row['label']}{hint}"
            assert row["gaps"] == gaps, f"{row['topic']}: expected gaps {gaps}, got {row['gaps']}"
            assert row["heading"] == heading, f"{row['topic']}: expected heading {heading!r}, got {row['heading']!r}"
    _run_check(5, check)


check_ex5()

### Exercise 6 — An isolated fan-out that survives a crashing subagent

Write `async def fan_out_isolated(jobs, deadline_s)`. `jobs` maps a subagent name to `(topic, work)`, where `work()` is a coroutine function returning a `SubagentResult`.

- Run all jobs **concurrently**, each under its **own** deadline (`asyncio.timeout(deadline_s)`).
- Return a list with **one `SubagentResult` per job, in the order of `jobs`**. Never raise.
- A job that misses its deadline → `status="failed"`, `failure_type="timeout"`, `error_category="transient"`, `retryable=True`.
- A job that raises any other exception → `status="failed"`, `failure_type="internal_error"`, `retryable=False`, and the exception's type name in `message`.

The checker calls it through `run_async(...)`, which works inside Jupyter. Theory: [README 10.8](README.md#108-timeouts-budgets-and-graceful-degradation-expanded-beyond-the-guide)

In [ ]:
async def fan_out_isolated(jobs: dict[str, tuple[str, Callable[[], Awaitable[SubagentResult]]]],
                           deadline_s: float) -> list[SubagentResult]:
    """Run every job concurrently, each under its own deadline; ALWAYS return one envelope per job, in order."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex6() -> None:
    def ok(name: str, delay: float):
        async def work():
            await asyncio.sleep(delay)
            return envelope("success", findings=2, subagent=name, topic="music")
        return work

    async def buggy():
        await asyncio.sleep(0.01)
        raise KeyError("source_url")

    async def hangs():
        await asyncio.sleep(5)
        return envelope("success", findings=1, subagent="news", topic="music")

    jobs = {"web": ("music", ok("web", 0.2)), "docs": ("music", ok("docs", 0.2)),
            "patents": ("music", buggy), "news": ("music", hangs), "blogs": ("music", ok("blogs", 0.2))}

    def check():
        t0 = time.perf_counter()
        out = run_async(fan_out_isolated(jobs, deadline_s=0.3))
        elapsed = time.perf_counter() - t0
        assert isinstance(out, list) and len(out) == 5, f"expected a list of 5 envelopes, got {out!r}"[:200]
        assert all(isinstance(r, SubagentResult) for r in out), \
            f"every item must be a SubagentResult, got {[type(r).__name__ for r in out]}"
        assert [r.subagent for r in out] == list(jobs), f"keep the job order {list(jobs)}, got {[r.subagent for r in out]}"
        assert [r.status for r in out] == ["success", "success", "failed", "failed", "success"], \
            f"statuses: expected success, success, failed, failed, success; got {[r.status for r in out]}"
        bug, slow = out[2], out[3]
        assert bug.failure_type == "internal_error" and bug.retryable is False and "KeyError" in (bug.message or ""), \
            f"the crashing job: expected internal_error, retryable=False and 'KeyError' in message, got {bug.failure_type}, {bug.retryable}, {bug.message!r}"
        assert slow.failure_type == "timeout" and slow.error_category == "transient" and slow.retryable is True, \
            f"the hanging job: expected timeout / transient / retryable=True, got {slow.failure_type}, {slow.error_category}, {slow.retryable}"
        assert elapsed < 0.7, f"took {elapsed:.2f}s: run the jobs concurrently (about 0.3s), each with its own deadline"
    _run_check(6, check)


check_ex6()

### Exercise 1b — classify a real API error (live, optional, ~$0, not scored)

Run it once Exercise 1 passes. An unknown model ID is rejected by the API (`404 not_found_error`), so nothing is billed.

In [ ]:
# (live, optional, ~$0, not scored) classify a REAL API error: an unknown model ID returns an error, which is not billed
if not RESULTS.get(1):
    print("Finish Exercise 1 first (its checker must pass), then run this cell.")
elif not HAS_KEY:
    print("skipped: no ANTHROPIC_API_KEY")
else:
    try:
        anthropic.Anthropic(max_retries=0).messages.create(
            model="claude-model-that-does-not-exist", max_tokens=50,
            messages=[{"role": "user", "content": "hi"}])
    except anthropic.APIError as err:
        print(type(err).__name__, getattr(err, "status_code", None), "->", classify_error(err),
              "| request_id:", getattr(err, "request_id", None))

## Part C — Architecture scenario

**Acquisition due-diligence pipeline.** An investment firm wants a multi-agent system that produces a due-diligence report on an acquisition target within **20 minutes** and a **$5** budget per report. A coordinator delegates to four subagents:

1. **Financial filings**: reads annual reports (PDFs; some are scanned, a few are corrupted).
2. **Litigation search**: queries a licensed court-records database through a service account whose credentials expire every 90 days.
3. **Patent portfolio**: queries a patent API that rate-limits aggressively at peak hours.
4. **News & sentiment**: searches news archives; one archive often returns `200 OK` with empty results during its nightly re-index.

The report goes to an investment committee that must know exactly which conclusions are solid. Describe the **error contract** of this system (15–25 lines):

1. The error taxonomy and the structured error payload your tools return (fields, examples for each category).
2. What each subagent recovers from locally, and what it propagates (with what context).
3. The subagent result envelope and the coordinator's recovery policy, including escalation.
4. Limits and isolation: deadlines, budgets, retries (and how you avoid retry amplification).
5. How the final report shows coverage and gaps, and what makes the system observable and resumable.

*Your answer here.*

<details><summary>Rubric: what a strong answer contains</summary>

- **Taxonomy and tool payloads**: four categories with a `retryable` flag that matches (transient → true; validation, business, permission → false as sent). Tool errors use `is_error` / `isError` plus `errorCategory`, `isRetryable`, a specific `message`, and `attemptedQuery` / partial results where relevant. Examples: patent 429 = transient; malformed court-records query = validation; expired service account = permission; a corrupted PDF = validation (not retryable as sent). The answer notes that these fields are a team convention inside the content, since MCP itself only defines `isError`.
- **Valid empty vs access failure**: "no litigation found" is a success with an empty list and is reported as a finding. The re-indexing news archive's empty `200 OK` is resolved locally (canary query, index freshness metadata) into either "empty" or "access failure", and never reaches the report as "no negative news".
- **Local recovery**: bounded retries with exponential backoff and jitter for timeouts and 429s (honoring `retry-after`), done inside the tool or subagent. Corrupted or scanned-unreadable PDFs, the expired credential and syntax errors are propagated immediately with context, not retried. A validation retry changes the request (reformulated query).
- **Envelope**: status (`success` / `empty` / `partial` / `failed`), failure type, category, retryable, attempted queries and attempt count, partial findings, alternatives, coverage impact, validated in the coordinator (Pydantic or similar). It lives in the subagent's final message (Agent SDK).
- **Coordinator policy**: accept, retry with a modified query, accept partial, delegate to an alternative source, escalate (the expired litigation credential goes to a human or ops queue right away, because litigation is business-critical for an acquisition), or continue and annotate the gap. It never aborts the whole report because of one source and never hides a failure.
- **Limits and isolation**: per-call timeouts and SDK `max_retries`; a per-subagent deadline (`asyncio.timeout` or `maxTurns`); a run deadline under 20 minutes and a cost cap (`max_budget_usd` in the Agent SDK, or task budgets plus `max_tokens` on the API); a retry budget; and retries at one layer only (the coordinator changes approach instead of repeating). Each subagent is wrapped so that it always returns an envelope. A circuit breaker protects against the rate-limited patent API.
- **Report**: coverage annotations computed in code against the **required** due-diligence checklist (not only against the subagents that ran), with FULL / PARTIAL / NONE labels and reasons per section, a "Coverage and limitations" list, and a code check that the synthesis kept the labels. A gap in a critical area such as litigation triggers human review before the report goes to the committee.
- **Observability and resume**: log every envelope and `request_id`; capture tool failures (for example the `PostToolUseFailure` hook); trend failure rates per source and type; persist envelopes in a manifest so a crashed run resumes the missing topics only; make writes idempotent.
</details>

## Scorecard

In [ ]:
quiz = SCORE.get("quiz", 0)
n_ex = 6
passed = sum(RESULTS.get(i, False) for i in range(1, n_ex + 1))
overall = (quiz + passed) / (10 + n_ex)
print(f"Quiz:       {quiz}/10 ({quiz / 10:.0%})")
print(f"Exercises:  {passed}/{n_ex} passed  " + " ".join(f"{i}:{'✓' if RESULTS.get(i) else '✗'}" for i in range(1, n_ex + 1)))
print(f"Overall:    {overall:.0%}  (pass mark 72%)")
print("PASS: on to Chapter 11" if overall >= 0.72 else "RETRY: reread the README sections behind your ✗ items, then try again")

Next chapter: [Chapter 11 — Context Management in Production Systems](../11-context-management/README.md)

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers with explanations</summary>

**Q1:** **B.** The query ran and matched nothing: that is a valid empty result, which is information ("no orders on record"). A and C mislabel a success as a failure, which causes pointless re-checks or retries. D is for protocol-level problems (unknown tool, malformed request), not for query outcomes.

**Q2:** **D.** A call cut off by a timeout is a transient failure that often succeeds on a second attempt, so the subagent should retry it locally (bounded, with backoff). A needs a converter or a human and B needs new credentials: neither is fixed by retrying, so they are propagated with context (and B escalated). C is a validation error: re-sending the identical query can never work.

**Q3:** **A.** The rejection is certain: the same query fails every time. A validation category with `isRetryable: false` stops identical retries, and naming the bad operator plus an accepted form gives the agent what it needs to change the request once. B invites more retries of a request that cannot succeed. C is silent suppression: the agent would report "no case law found". D retries at the wrong layer: the API call worked; the tool rejected the input.

**Q4:** **C.** The docs say a spend-cap 429 has no `retry-after`, carries `error.details.error_code: "enforced_spend_limit_reached"` (the documented way to tell it from a rate limit) and keeps failing until access resumes, so it behaves like a business condition, not a transient one. A burns calls and hides the cause. B misclassifies it and tries to work around an organizational control. D does not help: the spend cap applies to the organization, not to one model.

**Q5:** **B.** `max_turns` is a guardrail, not a completion signal. Hitting it is a failure to report together with what was completed, so the coordinator can choose a recovery. A presents an unfinished result as finished. C removes a safety cap. D throws away completed work and every other subagent's results.

**Q6:** **D.** In the Agent SDK a subagent returns only its final message to the coordinator. Error details written anywhere else never reach it, and a prose summary of a failure is easily read as "nothing found" (silent suppression by accident). A, B and C do not change what the coordinator receives.

**Q7:** **A.** The coordinator knows which topics were planned, which subagents ran and how each ended, so coverage is a fact computed in code (and can be checked again after synthesis). B lets an impression override facts. C cannot see topics that were never assigned. D hides the gaps the reader needs to judge confidence.

**Q8:** **C.** A timeout on a write does not tell you whether it happened, so retries are only safe for idempotent operations. A gives up cheap recoveries from real transient failures. B only makes the problem rarer. D cleans up the symptom after customers already saw it.

**Q9:** **D.** The integration layer worked (200s, `end_turn`) and the error contract was honest (the tools did return data). The answer contains a claim with no source, which is a model-output problem. A, B and C fix layers that did not fail.

**Q10:** **B.** A circuit breaker stops paying for timeouts on a source that is known to be failing, and graceful degradation keeps the run useful while the coverage annotation keeps it honest. A makes each failure slower. C hides the gap from readers. D causes retry amplification.
</details>

<details><summary>Exercise 1 — Classify Claude API errors: reference solution</summary>

```python
def classify_error(err: Exception) -> tuple[str, bool]:
    if isinstance(err, anthropic.APIConnectionError):          # includes APITimeoutError
        return "transient", True
    if isinstance(err, anthropic.APIStatusError):
        code = err.status_code
        if code == 429:
            body = err.body if isinstance(err.body, dict) else {}
            if (body.get("error") or {}).get("details", {}).get("error_code") == "enforced_spend_limit_reached":
                return "business", False                        # documented spend-cap signal
            return "transient", True                            # any other 429, with or without retry-after
        if code >= 500:                                          # 500, 503, 504, 529
            return "transient", True
        if code in (401, 403):
            return "permission", False
        if code == 402:
            return "business", False
        if code == 409:                                          # conflict: re-read state, then retry
            return "transient", True
        return "validation", False                              # 400, 404, 413, 422
    raise err
```
</details>

<details><summary>Exercise 2 — A retry helper that respects the category: reference solution</summary>

```python
import random


def retry_call(op: Callable[[], object], *, max_attempts: int = 3, base_delay: float = 0.5,
               max_delay: float = 8.0, sleep: Callable[[float], None] = time.sleep) -> tuple[object, int]:
    for attempt in range(1, max_attempts + 1):
        try:
            return op(), attempt
        except ToolFailure as err:
            if not err.retryable or attempt == max_attempts:
                err.attempts = attempt
                raise
            delay = min(base_delay * 2 ** (attempt - 1), max_delay)
            sleep(delay * random.uniform(0.5, 1.0))
    raise AssertionError("unreachable")
```
</details>

<details><summary>Exercise 3 — Access failure vs valid empty result: reference solution</summary>

```python
def search_outcome(source: str, query: str, backend: Callable[[str, str], list[dict]],
                   canary_query: str | None = None) -> SearchOutcome:
    def failed(error: str, retryable: bool) -> SearchOutcome:
        return SearchOutcome(source=source, query=query, status="failed", error=error, retryable=retryable)

    try:
        hits = backend(source, query)
    except TimeoutError as err:
        return failed(f"timeout: {err}", True)
    except PermissionError as err:
        return failed(f"permission denied: {err}", False)
    if hits:
        return SearchOutcome(source=source, query=query, status="ok", hits=hits)
    if canary_query is not None:
        try:
            canary_hits = backend(source, canary_query)
        except TimeoutError:
            canary_hits = []
        if not canary_hits:
            return failed("empty result and the canary query also returned nothing: source looks degraded", True)
    return SearchOutcome(source=source, query=query, status="empty")
```
</details>

<details><summary>Exercise 4 — The coordinator's recovery policy: reference solution</summary>

```python
def decide_v2(r: SubagentResult, retries_left: int, has_backup: bool) -> Action:
    if r.status in ("success", "empty"):
        return "accept"
    if r.error_category == "permission":
        return "escalate"
    can_retry = r.retryable or r.error_category == "validation"   # validation: retry only with a CHANGED request
    if can_retry and retries_left > 0 and r.alternatives:
        return "retry_modified"
    if r.status == "partial":
        return "accept_partial"
    if has_backup:
        return "delegate"
    return "annotate_gap"
```
</details>

<details><summary>Exercise 5 — Coverage annotations, computed in code: reference solution</summary>

```python
def coverage_report(required_topics: list[str], plan: dict[str, list[str]],
                    results: list[SubagentResult]) -> list[dict]:
    rows = []
    for topic in required_topics:
        if topic not in plan:
            label, gaps = "NONE", ["never planned"]
        else:
            got = [r for r in results if r.topic == topic]
            ran = {r.subagent for r in got}
            gaps = [f"{r.subagent}: {r.failure_type}" for r in got if r.status in ("partial", "failed")]
            gaps += [f"{s}: never ran" for s in plan[topic] if s not in ran]
            statuses = {r.status for r in got}
            if not got:
                label = "NONE"
            elif not gaps:
                label = "FULL"
            elif statuses & {"success", "empty", "partial"}:
                label = "PARTIAL"
            else:
                label = "NONE"
        suffix = {"FULL": "(FULL COVERAGE)", "PARTIAL": f"(PARTIAL COVERAGE: {'; '.join(gaps)})",
                  "NONE": f"(NO COVERAGE: {'; '.join(gaps)})"}[label]
        rows.append({"topic": topic, "label": label, "gaps": gaps, "heading": f"{topic} {suffix}"})
    return rows
```
</details>

<details><summary>Exercise 6 — An isolated fan-out that survives a crashing subagent: reference solution</summary>

```python
async def fan_out_isolated(jobs: dict[str, tuple[str, Callable[[], Awaitable[SubagentResult]]]],
                           deadline_s: float) -> list[SubagentResult]:
    async def one(name: str, topic: str, work) -> SubagentResult:
        try:
            async with asyncio.timeout(deadline_s):
                return await work()
        except TimeoutError:
            return SubagentResult(subagent=name, topic=topic, status="failed", failure_type="timeout",
                                  error_category="transient", retryable=True,
                                  message=f"no answer within {deadline_s}s", coverage_impact=f"{topic}: not covered")
        except Exception as exc:
            return SubagentResult(subagent=name, topic=topic, status="failed", failure_type="internal_error",
                                  retryable=False, message=f"{type(exc).__name__}: {exc}",
                                  coverage_impact=f"{topic}: not covered")

    return list(await asyncio.gather(*(one(name, topic, work) for name, (topic, work) in jobs.items())))
```
</details>